In [ ]:
from google.colab import drive
import os
import json
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "figures"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset restored.")
print("Rows:", len(feature_df))
print("Stations:", feature_df["StationId"].nunique())
print(
    "Date range:",
    feature_df["Datetime"].min(),
    "→",
    feature_df["Datetime"].max()
)

In [ ]:
!pip install -q sktime

In [ ]:
from sktime.transformations.vmd import VmdTransformer

print("VMD import successful.")

In [ ]:
unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("Train:")
print(
    train_df["Datetime"].min(),
    "→",
    train_df["Datetime"].max()
)

print("\nTest:")
print(
    test_df["Datetime"].min(),
    "→",
    test_df["Datetime"].max()
)

In [ ]:
station_counts = (
    train_df
    .groupby(
        ["StationId", "StationName"]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

top_stations = (
    station_counts
    .head(5)
    .reset_index()["StationId"]
    .tolist()
)

top_station_names = (
    train_df[
        train_df["StationId"].isin(
            top_stations
        )
    ][
        ["StationId", "StationName"]
    ]
    .drop_duplicates()
)

display(top_station_names)

In [ ]:
VMD_ALPHA = 2000
VMD_TAU = 0.0
VMD_K = 4
VMD_DC = 0
VMD_INIT = 1
VMD_TOL = 1e-7

print("VMD parameters:")
print("alpha:", VMD_ALPHA)
print("K:", VMD_K)
print("tau:", VMD_TAU)
print("tol:", VMD_TOL)

In [ ]:
def decompose_station_vmd(
    station_series,
    alpha=2000,
    K=4,
    tau=0.0,
    DC=0,
    init=1,
    tol=1e-7
):
    transformer = VmdTransformer(
        K=K,
        alpha=alpha,
        tau=tau,
        DC=DC,
        init=init,
        tol=tol,
        returned_decomp="u"
    )

    series = (
        station_series
        .astype(float)
        .reset_index(drop=True)
    )

    # VMD expects a univariate series.
    modes = transformer.fit_transform(
        series
    )

    return np.asarray(modes)

print("VMD decomposition function ready.")

In [ ]:
sample_station = top_stations[0]

sample_series = (
    train_df[
        train_df["StationId"] == sample_station
    ]
    .sort_values("Datetime")["AQI"]
    .dropna()
)

sample_modes = decompose_station_vmd(
    sample_series,
    alpha=VMD_ALPHA,
    K=VMD_K,
    tau=VMD_TAU,
    DC=VMD_DC,
    init=VMD_INIT,
    tol=VMD_TOL
)

print("Original series length:", len(sample_series))
print("VMD mode matrix shape:", sample_modes.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    sample_series.to_numpy(),
    label="Original AQI",
    linewidth=1
)

plt.title(
    f"Original AQI — Station {sample_station}"
)

plt.xlabel("Time index")
plt.ylabel("AQI")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "vmd_original_aqi.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

plt.figure(figsize=(14, 8))

for i in range(
    sample_modes.shape[1]
):
    plt.plot(
        sample_modes[:, i],
        label=f"Mode {i+1}"
    )

plt.title(
    f"VMD Modes — Station {sample_station}"
)

plt.xlabel("Time index")
plt.ylabel("Amplitude")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "vmd_modes.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
reconstructed = (
    sample_modes.sum(axis=1)
)

min_length = min(
    len(sample_series),
    len(reconstructed)
)

reconstruction_rmse = np.sqrt(
    np.mean(
        (
            sample_series.to_numpy()[:min_length]
            - reconstructed[:min_length]
        ) ** 2
    )
)

print(
    "VMD reconstruction RMSE:",
    reconstruction_rmse
)

In [ ]:
VMD_LOOKBACK = 12

print(
    "VMD sequence lookback:",
    VMD_LOOKBACK,
    "hours"
)

print(
    "Important:",
    "VMD will be fit on training-period station histories only."
)

In [ ]:
vmd_training_parts = []

for station_id in top_stations:

    station_train = (
        train_df[
            train_df["StationId"] == station_id
        ]
        .sort_values("Datetime")
        .copy()
    )

    station_train = station_train.dropna(
        subset=["AQI"]
    ).reset_index(drop=True)

    modes = decompose_station_vmd(
        station_train["AQI"],
        alpha=VMD_ALPHA,
        K=VMD_K,
        tau=VMD_TAU,
        DC=VMD_DC,
        init=VMD_INIT,
        tol=VMD_TOL
    )

    usable_length = min(
        len(station_train),
        modes.shape[0]
    )

    mode_df = station_train.iloc[
        :usable_length
    ][
        [
            "StationId",
            "StationName",
            "Datetime"
        ]
    ].copy()

    for mode_index in range(
        modes.shape[1]
    ):
        mode_df[
            f"VMD_Mode_{mode_index+1}"
        ] = modes[
            :usable_length,
            mode_index
        ]

    mode_df["AQI_t_plus_1"] = (
        station_train[
            "AQI_t_plus_1"
        ]
        .iloc[:usable_length]
        .to_numpy()
    )

    vmd_training_parts.append(
        mode_df
    )

vmd_train = pd.concat(
    vmd_training_parts,
    ignore_index=True
)

print(
    "VMD training dataset:",
    vmd_train.shape
)

display(
    vmd_train.head()
)

In [ ]:
def build_test_vmd_features(
    train_df,
    test_df,
    station_ids,
    lookback=12
):
    outputs = []

    for station_id in station_ids:

        full_station = pd.concat(
            [
                train_df[
                    train_df["StationId"] == station_id
                ],
                test_df[
                    test_df["StationId"] == station_id
                ]
            ]
        ).sort_values(
            "Datetime"
        ).drop_duplicates(
            ["StationId", "Datetime"]
        )

        test_times = (
            test_df[
                test_df["StationId"] == station_id
            ]["Datetime"]
            .sort_values()
            .tolist()
        )

        rows = []

        for current_time in test_times:

            history = full_station[
                full_station["Datetime"]
                <= current_time
            ].tail(48)

            if len(history) < lookback:
                continue

            aqis = (
                history["AQI"]
                .dropna()
                .to_numpy()
            )

            if len(aqis) < lookback:
                continue

            # Use only data available up to current_time.
            modes = decompose_station_vmd(
                aqis,
                alpha=VMD_ALPHA,
                K=VMD_K,
                tau=VMD_TAU,
                DC=VMD_DC,
                init=VMD_INIT,
                tol=VMD_TOL
            )

            recent_modes = modes[
                -lookback:
            ]

            rows.append(
                {
                    "StationId": station_id,
                    "Datetime": current_time,
                    "Modes": recent_modes
                }
            )

        outputs.extend(rows)

    return outputs

print("Test-side VMD function ready.")

In [ ]:
def decompose_station_vmd(
    station_series,
    alpha=2000,
    K=4,
    tau=0.0,
    DC=0,
    init=1,
    tol=1e-7
):
    transformer = VmdTransformer(
        K=K,
        alpha=alpha,
        tau=tau,
        DC=DC,
        init=init,
        tol=tol,
        returned_decomp="u"
    )

    # Accept either pandas Series or NumPy array
    if isinstance(station_series, pd.Series):
        series = (
            station_series
            .astype(float)
            .reset_index(drop=True)
        ).to_numpy()
    else:
        series = np.asarray(
            station_series,
            dtype=float
        ).reshape(-1)

    modes = transformer.fit_transform(
        series
    )

    return np.asarray(modes)


def build_test_vmd_features(
    train_df,
    test_df,
    station_ids,
    lookback=12
):
    outputs = []

    for station_id in station_ids:

        full_station = pd.concat(
            [
                train_df[
                    train_df["StationId"] == station_id
                ],
                test_df[
                    test_df["StationId"] == station_id
                ]
            ]
        ).sort_values(
            "Datetime"
        ).drop_duplicates(
            ["StationId", "Datetime"]
        )

        test_times = (
            test_df[
                test_df["StationId"] == station_id
            ]["Datetime"]
            .sort_values()
            .tolist()
        )

        rows = []

        for current_time in test_times:

            history = full_station[
                full_station["Datetime"] <= current_time
            ].tail(48)

            if len(history) < lookback:
                continue

            aqis = (
                history["AQI"]
                .dropna()
                .to_numpy(dtype=float)
            )

            if len(aqis) < lookback:
                continue

            modes = decompose_station_vmd(
                aqis,
                alpha=VMD_ALPHA,
                K=VMD_K,
                tau=VMD_TAU,
                DC=VMD_DC,
                init=VMD_INIT,
                tol=VMD_TOL
            )

            recent_modes = np.asarray(
                modes[-lookback:],
                dtype="float32"
            )

            rows.append(
                {
                    "StationId": station_id,
                    "Datetime": current_time,
                    "Modes": recent_modes
                }
            )

        outputs.extend(rows)

    return outputs


# Rebuild VMD test windows
vmd_test_records = build_test_vmd_features(
    train_df,
    test_df,
    top_stations,
    lookback=VMD_LOOKBACK
)

print(
    "VMD test prediction windows:",
    len(vmd_test_records)
)

In [ ]:
if len(vmd_test_records) > 0:

    X_vmd_test = np.asarray(
        [
            record["Modes"]
            for record in vmd_test_records
        ],
        dtype="float32"
    )

    print(
        "VMD test sequence shape:",
        X_vmd_test.shape
    )

    print(
        "Expected shape:",
        (
            len(vmd_test_records),
            VMD_LOOKBACK,
            VMD_K
        )
    )

else:
    print(
        "ERROR: No VMD test windows were generated."
    )

In [ ]:
X_vmd_test = np.asarray(
    [
        record["Modes"]
        for record in vmd_test_records
    ],
    dtype="float32"
)

print(
    "VMD test sequence shape:",
    X_vmd_test.shape
)

In [ ]:
X_vmd_train_list = []
y_vmd_train_list = []

for station_id in top_stations:

    station_data = (
        vmd_train[
            vmd_train["StationId"] == station_id
        ]
        .sort_values("Datetime")
        .reset_index(drop=True)
    )

    mode_columns = [
        f"VMD_Mode_{i+1}"
        for i in range(VMD_K)
    ]

    modes = station_data[
        mode_columns
    ].to_numpy(
        dtype="float32"
    )

    targets = station_data[
        "AQI_t_plus_1"
    ].to_numpy(
        dtype="float32"
    )

    for i in range(
        VMD_LOOKBACK - 1,
        len(station_data)
    ):

        X_vmd_train_list.append(
            modes[
                i - VMD_LOOKBACK + 1:i + 1
            ]
        )

        y_vmd_train_list.append(
            targets[i]
        )

X_vmd_train = np.asarray(
    X_vmd_train_list,
    dtype="float32"
)

y_vmd_train = np.asarray(
    y_vmd_train_list,
    dtype="float32"
)

print(
    "VMD training sequences:",
    X_vmd_train.shape
)

print(
    "VMD training targets:",
    y_vmd_train.shape
)

In [ ]:
# Use the last 15% of the VMD training sequences as validation.
vmd_split = int(
    len(X_vmd_train) * 0.85
)

X_vmd_train_fit = X_vmd_train[
    :vmd_split
]

y_vmd_train_fit = y_vmd_train[
    :vmd_split
]

X_vmd_val = X_vmd_train[
    vmd_split:
]

y_vmd_val = y_vmd_train[
    vmd_split:
]

print(
    "VMD fit sequences:",
    X_vmd_train_fit.shape
)

print(
    "VMD validation sequences:",
    X_vmd_val.shape
)

In [ ]:
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

print("TensorFlow version:", tf.__version__)
print(
    "GPU available:",
    len(tf.config.list_physical_devices("GPU")) > 0
)

vmd_lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(
            VMD_LOOKBACK,
            VMD_K
        ),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

vmd_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

vmd_lstm_model.summary()

In [ ]:
vmd_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

vmd_history = vmd_lstm_model.fit(
    X_vmd_train_fit,
    y_vmd_train_fit,
    validation_data=(
        X_vmd_val,
        y_vmd_val
    ),
    epochs=30,
    batch_size=256,
    callbacks=vmd_callbacks,
    verbose=1
)

print("VMD-LSTM training completed.")

In [ ]:
# Build matching actual target values from the test windows.
vmd_test_targets = []

for record in vmd_test_records:

    row = test_df[
        (test_df["StationId"] == record["StationId"])
        &
        (test_df["Datetime"] == record["Datetime"])
    ]

    if len(row) == 1:
        vmd_test_targets.append(
            row["AQI_t_plus_1"].iloc[0]
        )
    else:
        vmd_test_targets.append(
            np.nan
        )

vmd_test_targets = np.asarray(
    vmd_test_targets,
    dtype="float32"
)

valid_mask = np.isfinite(
    vmd_test_targets
)

X_vmd_test_valid = X_vmd_test[
    valid_mask
]

y_vmd_test_valid = vmd_test_targets[
    valid_mask
]

vmd_pred = (
    vmd_lstm_model
    .predict(
        X_vmd_test_valid,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

vmd_mae = mean_absolute_error(
    y_vmd_test_valid,
    vmd_pred
)

vmd_rmse = np.sqrt(
    mean_squared_error(
        y_vmd_test_valid,
        vmd_pred
    )
)

vmd_r2 = r2_score(
    y_vmd_test_valid,
    vmd_pred
)

print("VMD-LSTM")
print("--------")
print("MAE :", round(vmd_mae, 6))
print("RMSE:", round(vmd_rmse, 6))
print("R²  :", round(vmd_r2, 6))

In [ ]:
!pip install -q pyswarms

In [ ]:
from pyswarms.single import GlobalBestPSO

# Use one representative high-coverage training station
pso_station_id = top_stations[0]

pso_series = (
    train_df[
        train_df["StationId"] == pso_station_id
    ]
    .sort_values("Datetime")["AQI"]
    .dropna()
    .to_numpy()
)

# Use a manageable recent training segment
MAX_PSO_POINTS = 10000

if len(pso_series) > MAX_PSO_POINTS:
    pso_series = pso_series[
        -MAX_PSO_POINTS:
    ]

pso_split = int(
    len(pso_series) * 0.85
)

pso_fit_series = pso_series[
    :pso_split
]

pso_validation_series = pso_series[
    pso_split:
]

print(
    "PSO fitting points:",
    len(pso_fit_series)
)

print(
    "PSO validation points:",
    len(pso_validation_series)
)

In [ ]:
def vmd_pso_objective(
    particles
):
    costs = []

    for particle in particles:

        alpha = float(
            particle[0]
        )

        K = int(
            round(particle[1])
        )

        K = max(
            2,
            min(K, 6)
        )

        try:

            transformer = VmdTransformer(
                K=K,
                alpha=alpha,
                tau=0.0,
                DC=0,
                init=1,
                tol=1e-6,
                returned_decomp="u"
            )

            modes = transformer.fit_transform(
                pso_fit_series
            )

            reconstruction = np.asarray(
                modes
            ).sum(axis=1)

            usable = min(
                len(reconstruction),
                len(pso_fit_series)
            )

            train_reconstruction_error = np.mean(
                (
                    pso_fit_series[:usable]
                    - reconstruction[:usable]
                ) ** 2
            )

            costs.append(
                train_reconstruction_error
            )

        except Exception:
            costs.append(
                1e12
            )

    return np.asarray(costs)

print("PSO objective ready.")

In [ ]:
bounds = (
    np.array([
        500.0,
        2.0
    ]),
    np.array([
        5000.0,
        6.0
    ])
)

options = {
    "c1": 0.5,
    "c2": 0.3,
    "w": 0.8
}

optimizer = GlobalBestPSO(
    n_particles=8,
    dimensions=2,
    options=options,
    bounds=bounds
)

print("PSO configured.")

In [ ]:
best_cost, best_position = optimizer.optimize(
    vmd_pso_objective,
    iters=8,
    verbose=True
)

best_alpha = float(
    best_position[0]
)

best_K = int(
    round(best_position[1])
)

best_K = max(
    2,
    min(best_K, 6)
)

print("Best alpha:", best_alpha)
print("Best K:", best_K)
print("Best objective:", best_cost)

In [ ]:
standard_transformer = VmdTransformer(
    K=VMD_K,
    alpha=VMD_ALPHA,
    tau=VMD_TAU,
    DC=VMD_DC,
    init=VMD_INIT,
    tol=VMD_TOL,
    returned_decomp="u"
)

standard_modes = standard_transformer.fit_transform(
    pso_fit_series
)

standard_reconstruction = np.asarray(
    standard_modes
).sum(axis=1)

standard_error = np.sqrt(
    np.mean(
        (
            pso_fit_series
            - standard_reconstruction
        ) ** 2
    )
)

pso_transformer = VmdTransformer(
    K=best_K,
    alpha=best_alpha,
    tau=0.0,
    DC=0,
    init=1,
    tol=1e-6,
    returned_decomp="u"
)

pso_modes = pso_transformer.fit_transform(
    pso_fit_series
)

pso_reconstruction = np.asarray(
    pso_modes
).sum(axis=1)

pso_error = np.sqrt(
    np.mean(
        (
            pso_fit_series
            - pso_reconstruction
        ) ** 2
    )
)

print("Standard VMD reconstruction RMSE:", standard_error)
print("PSO-VMD reconstruction RMSE:", pso_error)

In [ ]:
vmd_results = pd.DataFrame({
    "Model": [
        "VMD-LSTM"
    ],
    "MAE": [
        vmd_mae
    ],
    "RMSE": [
        vmd_rmse
    ],
    "R2": [
        vmd_r2
    ]
})

vmd_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase9_vmd_results.csv"
    ),
    index=False
)

pso_results = pd.DataFrame({
    "Parameter": [
        "Standard Alpha",
        "Standard K",
        "PSO Alpha",
        "PSO K",
        "Standard Reconstruction RMSE",
        "PSO Reconstruction RMSE"
    ],
    "Value": [
        VMD_ALPHA,
        VMD_K,
        best_alpha,
        best_K,
        standard_error,
        pso_error
    ]
})

pso_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase9_pso_vmd_results.csv"
    ),
    index=False
)

print("Phase 9 result tables saved.")

In [ ]:
vmd_lstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase9_vmd_lstm.keras"
    )
)

print("VMD-LSTM model saved.")

In [ ]:
phase9_summary = {
    "completed_phase": "Phase 9 - VMD and PSO-VMD Experiments",
    "next_phase": "Phase 10 - TimeGAN / Advanced Hybrid Experiments",

    "selected_stations": [
        str(station)
        for station in top_stations
    ],

    "standard_vmd": {
        "alpha": VMD_ALPHA,
        "K": VMD_K
    },

    "vmd_lstm": {
        "MAE": float(vmd_mae),
        "RMSE": float(vmd_rmse),
        "R2": float(vmd_r2)
    },

    "pso_vmd": {
        "alpha": float(best_alpha),
        "K": int(best_K),
        "standard_reconstruction_rmse": float(
            standard_error
        ),
        "pso_reconstruction_rmse": float(
            pso_error
        )
    }
}

phase9_summary_path = os.path.join(
    OUTPUT_PATH,
    "phase9_summary.json"
)

with open(
    phase9_summary_path,
    "w"
) as f:
    json.dump(
        phase9_summary,
        f,
        indent=4
    )

print("Phase 9 summary saved.")

In [ ]:
print("=" * 60)
print("PHASE 9 — VMD / PSO-VMD SUMMARY")
print("=" * 60)

print("Stations used:")
print(top_station_names)

print("\nStandard VMD:")
print("Alpha:", VMD_ALPHA)
print("K:", VMD_K)

print("\nVMD-LSTM:")
print("MAE :", round(vmd_mae, 6))
print("RMSE:", round(vmd_rmse, 6))
print("R²  :", round(vmd_r2, 6))

print("\nPSO-VMD:")
print("Best alpha:", best_alpha)
print("Best K:", best_K)
print("Standard reconstruction RMSE:", round(standard_error, 6))
print("PSO reconstruction RMSE:", round(pso_error, 6))

print("=" * 60)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save VMD-LSTM model
# --------------------------------------------------

vmd_lstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase9_vmd_lstm.keras"
    )
)

# --------------------------------------------------
# 2. Save VMD results
# --------------------------------------------------

vmd_results = pd.DataFrame({
    "Model": ["VMD-LSTM"],
    "MAE": [vmd_mae],
    "RMSE": [vmd_rmse],
    "R2": [vmd_r2]
})

vmd_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase9_vmd_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 3. Save PSO-VMD results
# --------------------------------------------------

pso_results = pd.DataFrame({
    "Parameter": [
        "Standard Alpha",
        "Standard K",
        "PSO Alpha",
        "PSO K",
        "Standard Reconstruction RMSE",
        "PSO Reconstruction RMSE"
    ],
    "Value": [
        VMD_ALPHA,
        VMD_K,
        best_alpha,
        best_K,
        standard_error,
        pso_error
    ]
})

pso_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase9_pso_vmd_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 4. Save Phase 9 summary
# --------------------------------------------------

phase9_summary = {
    "completed_phase": "Phase 9 - VMD and PSO-VMD Experiments",
    "next_phase": "Phase 10 - TimeGAN and Advanced Hybrid Experiments",

    "stations_used": [
        {
            "StationId": str(row["StationId"]),
            "StationName": str(row["StationName"])
        }
        for _, row in top_station_names.iterrows()
    ],

    "standard_vmd": {
        "alpha": float(VMD_ALPHA),
        "K": int(VMD_K)
    },

    "vmd_lstm": {
        "MAE": float(vmd_mae),
        "RMSE": float(vmd_rmse),
        "R2": float(vmd_r2)
    },

    "pso_vmd": {
        "best_alpha": float(best_alpha),
        "best_K": int(best_K),
        "standard_reconstruction_rmse": float(
            standard_error
        ),
        "pso_reconstruction_rmse": float(
            pso_error
        )
    }
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase9_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        phase9_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 9 - VMD and PSO-VMD Experiments",
    "next_phase": "Phase 10 - TimeGAN and Advanced Hybrid Experiments",
    "best_vmd_forecasting_result": "VMD-LSTM",
    "pso_vmd_reconstruction_improved": True
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_path, "w") as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 6. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    os.path.join(
        MODEL_PATH,
        "phase9_vmd_lstm.keras"
    ),
    os.path.join(
        TABLE_PATH,
        "phase9_vmd_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase9_pso_vmd_results.csv"
    ),
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 9 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 9")
print("Next: Phase 10 - TimeGAN and Advanced Hybrid Experiments")